In [22]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import math

In [23]:
#1.Positional Encoding

class PositionalEncoding(nn.Module):
    def __init__(self,d_model,max_len=100):
        super().__init__()

        pe = torch.zeros(max_len,d_model)

        for pos in range(max_len):
            for i in range(0,d_model,2):

                pe[pos,i] = math.sin(pos/(10000 ** ((2*i)/d_model)))

                if i + 1 < d_model :
                    pe[pos, i+1] = math.cos(pos / (10000 ** ((2*i)/d_model)))
        
        self.pe = pe.unsqueeze(0)


    def forward(self,x):
        x = x + self.pe[:,:x.size(1)]
        return x

In [24]:
#scaled dot product attention 
def attention (Q,K,V):
    d_k = Q.size(-1)
    scores = torch.matmul(Q,K.transpose(-2,-2)/math.sqrt(d_k))
    weights = F.softmax(scores,dim = -1)
    output = torch.matmul(weights,V)
    return output

In [ ]:
#Multi Head attention

class MultiHeadAttention(nn.Module):

    def __init__(self,d_model,heads):
        super().__init__()
 
        self.heads = heads
        self.d_k  = d_model // heads

        self.q_linear = nn.Linear(d_model,d_model)
        self.k_linear = nn.Linear(d_model,d_model)
        self.v_linear = nn.Linear(d_model,d_model)

        self.fc_out = nn.Linear(d_model,d_model)

    
    
    



In [26]:
class FeedForward(nn.Module):   #Feed forward neural network is added

    def __init__(self, d_model):
        super().__init__()

        self.fc1 = nn.Linear(d_model, 4 * d_model)
        self.fc2 = nn.Linear(4 * d_model, d_model)

    def forward(self, x):

        x = F.relu(self.fc1(x))

        x = self.fc2(x)

        return x

In [27]:
#Transfomer Block
class TransformerBlock(nn.Module):

    def __init__(self, d_model, heads):
        super().__init__()

        self.attention = MultiHeadAttention(d_model, heads)

        self.norm1 = nn.LayerNorm(d_model)

        self.feed_forward = FeedForward(d_model)

        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):

        attn_output = self.attention(x)

        x = self.norm1(x + attn_output)

        ff_output = self.feed_forward(x)

        x = self.norm2(x + ff_output)

        return x

In [28]:
class TransformerBlock(nn.Module):

    def __init__(self, d_model, heads):
        super().__init__()

        self.attention = MultiHeadAttention(d_model, heads)

        self.norm1 = nn.LayerNorm(d_model)

        self.feed_forward = FeedForward(d_model)

        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):

        attn_output = self.attention(x)

        x = self.norm1(x + attn_output)

        ff_output = self.feed_forward(x)

        x = self.norm2(x + ff_output)

        return x

In [29]:
class MiniTransformer(nn.Module):

    def __init__(self, vocab_size, d_model, heads, num_layers):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)

        self.position = PositionalEncoding(d_model)

        self.layers = nn.ModuleList(
            [TransformerBlock(d_model, heads) for _ in range(num_layers)]
        )

        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):

        x = self.embedding(x)

        x = self.position(x)

        for layer in self.layers:
            x = layer(x)

        output = self.fc_out(x)

        return output

In [31]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


# ----------------------------
# 1. Positional Encoding
# ----------------------------

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=100):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        for pos in range(max_len):
            for i in range(0, d_model, 2):

                pe[pos, i] = math.sin(pos / (10000 ** ((2*i)/d_model)))

                if i + 1 < d_model:
                    pe[pos, i+1] = math.cos(pos / (10000 ** ((2*i)/d_model)))

        # register as buffer so it moves with model
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):

        x = x + self.pe[:, :x.size(1)]

        return x


# ----------------------------
# 2. Scaled Dot Product Attention
# ----------------------------

def attention(Q, K, V):

    d_k = Q.size(-1)

    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    weights = F.softmax(scores, dim=-1)

    output = torch.matmul(weights, V)

    return output


# ----------------------------
# 3. Multi Head Attention
# ----------------------------

class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, heads):
        super().__init__()

        self.heads = heads
        self.d_k = d_model // heads

        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)

        self.fc_out = nn.Linear(d_model, d_model)

    def forward(self, x):

        batch_size, seq_len, d_model = x.shape

        # compute Q K V
        Q = self.q_linear(x)
        K = self.k_linear(x)
        V = self.v_linear(x)

        # split heads
        Q = Q.view(batch_size, seq_len, self.heads, self.d_k)
        K = K.view(batch_size, seq_len, self.heads, self.d_k)
        V = V.view(batch_size, seq_len, self.heads, self.d_k)

        # rearrange dimensions
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        # attention
        out = attention(Q, K, V)

        # combine heads
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)

        out = self.fc_out(out)

        return out


# ----------------------------
# 4. Feed Forward Network
# ----------------------------

class FeedForward(nn.Module):

    def __init__(self, d_model):
        super().__init__()

        self.fc1 = nn.Linear(d_model, 4 * d_model)
        self.fc2 = nn.Linear(4 * d_model, d_model)

    def forward(self, x):

        x = F.relu(self.fc1(x))

        x = self.fc2(x)

        return x


# ----------------------------
# 5. Transformer Block
# ----------------------------

class TransformerBlock(nn.Module):

    def __init__(self, d_model, heads):
        super().__init__()

        self.attention = MultiHeadAttention(d_model, heads)

        self.norm1 = nn.LayerNorm(d_model)

        self.feed_forward = FeedForward(d_model)

        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):

        attn_output = self.attention(x)

        x = self.norm1(x + attn_output)

        ff_output = self.feed_forward(x)

        x = self.norm2(x + ff_output)

        return x


# ----------------------------
# 6. Full Transformer Model
# ----------------------------

class MiniTransformer(nn.Module):

    def __init__(self, vocab_size, d_model, heads, num_layers):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)

        self.position = PositionalEncoding(d_model)

        self.layers = nn.ModuleList(
            [TransformerBlock(d_model, heads) for _ in range(num_layers)]
        )

        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):

        x = self.embedding(x)

        x = self.position(x)

        for layer in self.layers:
            x = layer(x)

        output = self.fc_out(x)

        return output


# ----------------------------
# 7. Example Run
# ----------------------------

vocab_size = 50
d_model = 32
heads = 4
num_layers = 2

model = MiniTransformer(vocab_size, d_model, heads, num_layers)

x = torch.randint(0, vocab_size, (1, 10))

output = model(x)

print(output.shape)

torch.Size([1, 10, 50])
